# Question answering from one PDF

Upload **one text-based PDF**, then type a question about it. The tool extracts text page by page, cleans it, splits long pages into passages, ranks passages against the question, and uses a pretrained extractive QA model to return an answer with its page. Run the cells from top to bottom in Google Colab. Re-run the question cell to ask another question about the same PDF; re-run the upload cell to change PDFs.



## 1. Install and load the model

In [1]:
# Install the libraries used to read PDFs, rank passages, and answer questions.
!pip install -q pymupdf "transformers==4.57.1" scikit-learn

import re
from pathlib import Path
import pymupdf
from google.colab import files
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

# Load a pretrained extractive QA model; it selects text spans from a passage.
qa_v2 = pipeline("question-answering", model="deepset/minilm-uncased-squad2")
print("Question-answering model is ready")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 56.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/477 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Some weights of the model checkpoint at deepset/minilm-uncased-squad2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/107 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cpu


Question-answering model is ready


## 2. Upload one source PDF

In [2]:
# Select one PDF to use for all questions until this cell is run again.
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Please select exactly one PDF and rerun this cell.")
PDF_PATH = next(iter(uploaded))
if Path(PDF_PATH).suffix.lower() != ".pdf":
    raise ValueError("Please upload a PDF file.")

# Confirm that this PDF has pages and extractable text.
with pymupdf.open(PDF_PATH) as doc:
    page_count = len(doc)
    readable_characters = sum(len(page.get_text("text").strip()) for page in doc)
print(f"Selected PDF: {PDF_PATH} ({page_count} pages, {readable_characters} extracted characters)")
if not readable_characters:
    print("No selectable text was found. A scanned PDF needs OCR before this tool can answer it.")


Saving Test PDF (Hard).pdf to Test PDF (Hard).pdf
Selected PDF: Test PDF (Hard).pdf (3 pages, 4780 extracted characters)


## 3. Clean, split, and rank PDF text

In [3]:
def clean_text(text):
    # Remove long runs of decorative symbols found in some PDF extractions.
    text = re.sub(r"(?:[^\w\s]\s*){8,}", " ", text)
    # Replace repeated whitespace and line breaks with single spaces.
    return re.sub(r"\s+", " ", text).strip()


def chunk_text(text, max_tokens=300, overlap_tokens=50):
    # Split long text to fit the QA model, repeating a little context at boundaries.
    if not 0 <= overlap_tokens < max_tokens:
        raise ValueError("overlap_tokens must be nonnegative and smaller than max_tokens")
    token_ids = qa_v2.tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    start = 0
    while start < len(token_ids):
        end = min(start + max_tokens, len(token_ids))
        chunks.append(qa_v2.tokenizer.decode(token_ids[start:end], skip_special_tokens=True))
        if end == len(token_ids):
            break
        start = end - overlap_tokens
    return chunks


def rank_chunks(question, chunks):
    # Rank passages by word overlap with the question before asking the QA model.
    if not chunks:
        return []
    vectors = TfidfVectorizer(stop_words="english").fit_transform([question] + chunks)
    relevance = cosine_similarity(vectors[0:1], vectors[1:])[0]
    return sorted(range(len(chunks)), key=lambda index: relevance[index], reverse=True)


## 4. Answer a question and cite its page

In [4]:
def answer_from_pdf(pdf_path, question):
    # Remove list numbering such as "4. " or "7)" before retrieval and QA.
    question = re.sub(r"^\s*\d+[.)](?=\s|[A-Za-z])\s*", "", question).strip()
    if not question:
        return "Please enter a question."

    # Keep each passage paired with the page where it appeared.
    chunks_with_pages = []
    with pymupdf.open(pdf_path) as doc:
        for page_number, page in enumerate(doc, start=1):
            page_text = clean_text(page.get_text("text"))
            for chunk in chunk_text(page_text):
                chunks_with_pages.append((chunk, page_number))

    if not chunks_with_pages:
        return f"No readable text found in: {pdf_path}"

    # Try more relevant passages first, using an empty model answer as a rejection.
    chunk_texts = [chunk for chunk, _ in chunks_with_pages]
    for index in rank_chunks(question, chunk_texts):
        chunk, page_number = chunks_with_pages[index]
        result = qa_v2(question=question, context=chunk, handle_impossible_answer=True)
        if result["answer"].strip():
            return f"Answer: {result['answer']}\nSource: {Path(pdf_path).name}, page {page_number}"

    # This fallback is imperfect: the model can also give a wrong nonempty answer.
    return f"Answer not available in the PDF: {Path(pdf_path).name}"


## 5. Ask your own question

In [5]:
# Re-run this cell to ask another question without uploading the PDF again.
user_question = input("Ask a question about the selected PDF: ").strip()
print(answer_from_pdf(PDF_PATH, user_question))


Ask a question about the selected PDF: What is the capital city of Bangladesh?
Answer: dhaka
Source: Test PDF (Hard).pdf, page 1


## 6. Evaluation

The two supplied Bangladesh PDFs previously yielded **7/7 answerable questions each**. A separate fictional seagrass PDF yielded **8/9 answerable questions** when question 4 was prefixed by a list number: the model returned 600 rather than 4,800.

The absent national-bird question previously returned “royal bengal tiger,” which the PDF identifies as the national animal. A no-answer response is therefore **not reliable for closely related missing facts**. The budget question on the seagrass PDF was rejected correctly. More robust answer validation would be needed before relying on the no-answer response.

In [6]:
# Run the matching question set for the currently selected test PDF.
bangladesh_questions = [
    "What is the capital city of Bangladesh?",
    "Which river is considered the lifeline of Bangladesh?",
    "In which year did Bangladesh gain independence?",
    "What is the official language of Bangladesh?",
    "What is the largest mangrove forest in the world?",
    "What is the currency of Bangladesh?",
    "What is the national animal of Bangladesh?",
]
seagrass_questions = [
    "1. Which organization led the pilot?",
    "2. Where was the pilot conducted?",
    "3. When did planting begin?",
    "4. How many seedlings were initially planted?",
    "5. Which device measured turbidity?",
    "6. Which plot had higher survival after six months?",
    "7. What damaged seedlings in the Western Plot?",
    "8. How many seedlings were moved from the Western Plot?",
    "9. When was the follow-up survey scheduled?",
    "10. What was the pilot’s total budget?",
]

pdf_name = Path(PDF_PATH).name.lower()
if "test pdf (easy)" in pdf_name or "test pdf (hard)" in pdf_name:
    test_questions = bangladesh_questions
elif "cala_bay_seagrass" in pdf_name:
    test_questions = seagrass_questions
else:
    test_questions = []

if not test_questions:
    print("No prepared questions for this PDF. Ask your own question in section 5.")
else:
    print("Testing PDF:", PDF_PATH)
    for number, question in enumerate(test_questions, start=1):
        print(f"{number}. {question}")
        print(answer_from_pdf(PDF_PATH, question))
        print()


Testing PDF: Test PDF (Hard).pdf
1. What is the capital city of Bangladesh?
Answer: dhaka
Source: Test PDF (Hard).pdf, page 1

2. Which river is considered the lifeline of Bangladesh?
Answer: padma river
Source: Test PDF (Hard).pdf, page 1

3. In which year did Bangladesh gain independence?
Answer: 1971
Source: Test PDF (Hard).pdf, page 1

4. What is the official language of Bangladesh?
Answer: bengali
Source: Test PDF (Hard).pdf, page 1

5. What is the largest mangrove forest in the world?
Answer: sundarbans
Source: Test PDF (Hard).pdf, page 2

6. What is the currency of Bangladesh?
Answer: bangladeshi taka
Source: Test PDF (Hard).pdf, page 2

7. What is the national animal of Bangladesh?
Answer: royal bengal tiger
Source: Test PDF (Hard).pdf, page 2



### Important negative test

With either supplied Bangladesh PDF selected, the following question asks for a fact absent from that PDF. Earlier runs returned the **national animal** incorrectly.

In [7]:
# Test a closely related question whose answer is absent from the Bangladesh PDF.
if "test pdf (easy)" in pdf_name or "test pdf (hard)" in pdf_name:
    print(answer_from_pdf(PDF_PATH, "What is the national bird of Bangladesh?"))
else:
    print("Select an Easy or Hard Bangladesh PDF to run the national-bird negative test.")


Answer: royal bengal tiger
Source: Test PDF (Hard).pdf, page 2
